In [0]:
from pyspark.sql import functions as F

In [0]:
%sql
create table if not exists brz_bank_transactions
using csv
options (
  header = "true",
  inferSchema = "true"
)
location 'abfss://project-dataset@finrisk3605359934826.dfs.core.windows.net/';

## Data Auditing

In [0]:
table_name = "brz_bank_transactions"
audit_df = spark.table(table_name)
row_count = audit_df.count()
column_count = len(audit_df.columns)

print(f"Auditing table: {table_name}")
print(f"Row count: {row_count:,}")
print(f"Column count: {column_count}")
display(audit_df.limit(5))


In [0]:
numeric_features = ["transaction_amount", "account_balance", "credit_score", "has_loan", 
                    "emi_amount", "transaction_hour"]
categorical_features = ["account_type", "transaction_type","merchant_category","transaction_status"
                         "state", "loan_type", "channel", "kyc_status","transaction_direction"]
datetime_feature = ["transaction_date", "transaction_time"]
identifiers = ["transaction_id", "customer_id"]
target_feature = ["is_fraud"]

print(f"numeric feature count: {len(numeric_features)}")
print(f"categorical feature count: {len(categorical_features)}")
print(f"datetime feature count: {len(datetime_feature)}")
print(f"identifier count: {len(identifiers)}")


In [0]:
# Schema checks
# Verifying that the bronze table still matches the expected structure
# and highlights any missing, extra, or mistyped columns.

expected_schema = {
    "transaction_id": "string",
    "customer_id": "string",
    "transaction_date": "date",
    "transaction_time": "timestamp",
    "account_type": "string",
    "transaction_type": "string",
    "transaction_amount": "double",
    "transaction_direction": "string",
    "account_balance": "double",
    "merchant_category": "string",
    "state": "string",
    "credit_score": "int",
    "has_loan": "int",
    "loan_type": "string",
    "emi_amount": "double",
    "transaction_status": "string",
    "channel": "string",
    "kyc_status": "string",
    "is_fraud": "int",
    "transaction_hour": "int",
}

actual_schema = {field.name: field.dataType.simpleString() for field in audit_df.schema.fields}

schema_rows = [
    (
        column_name,
        expected_schema[column_name],
        actual_schema.get(column_name),
        expected_schema[column_name] == actual_schema.get(column_name),
    )
    for column_name in expected_schema
]

missing_columns = sorted(set(expected_schema) - set(actual_schema))
unexpected_columns = sorted(set(actual_schema) - set(expected_schema))

schema_check_df = spark.createDataFrame(
    schema_rows,
    ["column_name", "expected_type", "actual_type", "matches_expected_type"],
)

print(f"Missing expected columns: {missing_columns or 'None'}")
print(f"Unexpected columns: {unexpected_columns or 'None'}")
display(schema_check_df.orderBy(F.col("matches_expected_type").asc(), F.col("column_name").asc()))


In [0]:
# Missing value checks
# Strings are treated as missing when they are null or blank after trimming whitespace.

if row_count == 0:
    print("The table is empty, so missing value profiling cannot be computed.")
else:
    missing_exprs = []
    for column_name, data_type in audit_df.dtypes:
        if data_type == "string":
            condition = F.col(column_name).isNull() | (F.trim(F.col(column_name)) == "")
        else:
            condition = F.col(column_name).isNull()

        missing_exprs.append(
            F.sum(F.when(condition, 1).otherwise(0)).alias(column_name)
        )

    missing_summary = audit_df.agg(*missing_exprs)
    stack_expr = "stack({0}, {1}) as (column_name, missing_count)".format(
        len(audit_df.columns),
        ", ".join([f"'{column_name}', `{column_name}`" for column_name in audit_df.columns]),
    )

    missing_long = (
        missing_summary
        .selectExpr(stack_expr)
        .withColumn("missing_pct", F.round(F.col("missing_count") / F.lit(row_count) * 100, 2))
        .orderBy(F.desc("missing_count"), F.asc("column_name"))
    )

    display(missing_long)


In [0]:
# Duplicate checks
# Full-row duplicates indicate exact record duplication.
# Transaction ID duplicates indicate business-key duplication.

if row_count == 0:
    print("The table is empty, so duplicate checks cannot be computed.")
else:
    full_row_duplicate_count = row_count - audit_df.dropDuplicates().count()

    transaction_id_duplicates = (
        audit_df.groupBy("transaction_id")
        .count()
        .filter(F.col("count") > 1)
        .orderBy(F.desc("count"), F.asc("transaction_id"))
    )

    duplicate_transaction_ids = transaction_id_duplicates.count()

    print(f"Exact duplicate rows: {full_row_duplicate_count:,}")
    print(f"Duplicate transaction_id values: {duplicate_transaction_ids:,}")
    display(transaction_id_duplicates.limit(20))


In [0]:
# Invalid value checks
# These rules focus on clear business or data-quality violations and cross-field inconsistencies.

invalid_rules = [
    ("negative_transaction_amount", F.col("transaction_amount") < 0),
    ("negative_account_balance_potential_issue", F.col("account_balance") < 0),
    (
        "credit_score_outside_300_900",
        F.col("credit_score").isNotNull() & ((F.col("credit_score") < 300) | (F.col("credit_score") > 900)),
    ),
    (
        "transaction_hour_outside_0_23",
        F.col("transaction_hour").isNotNull() & ((F.col("transaction_hour") < 0) | (F.col("transaction_hour") > 23)),
    ),
    (
        "has_loan_not_binary",
        F.col("has_loan").isNotNull() & (~F.col("has_loan").isin(0, 1)),
    ),
    (
        "is_fraud_not_binary",
        F.col("is_fraud").isNotNull() & (~F.col("is_fraud").isin(0, 1)),
    ),
    (
        "unexpected_transaction_direction",
        F.col("transaction_direction").isNotNull() & (~F.col("transaction_direction").isin("Debit", "Credit")),
    ),
    (
        "loan_type_inconsistent_with_has_loan",
        ((F.col("has_loan") == 0) & F.col("loan_type").isNotNull() & (F.trim(F.lower(F.col("loan_type"))) != "none"))
        | ((F.col("has_loan") == 1) & (F.col("loan_type").isNull() | (F.trim(F.lower(F.col("loan_type"))) == "none"))),
    ),
    (
        "emi_amount_inconsistent_with_has_loan",
        ((F.col("has_loan") == 0) & (F.col("emi_amount") > 0))
        | ((F.col("has_loan") == 1) & (F.col("emi_amount").isNull() | (F.col("emi_amount") <= 0))),
    ),
]

if row_count == 0:
    print("The table is empty, so invalid value checks cannot be computed.")
else:
    invalid_summary = audit_df.agg(
        *[F.sum(F.when(condition, 1).otherwise(0)).alias(rule_name) for rule_name, condition in invalid_rules]
    )

    stack_expr = "stack({0}, {1}) as (rule_name, invalid_count)".format(
        len(invalid_rules),
        ", ".join([f"'{rule_name}', `{rule_name}`" for rule_name, _ in invalid_rules]),
    )

    invalid_long = (
        invalid_summary
        .selectExpr(stack_expr)
        .withColumn("invalid_pct", F.round(F.col("invalid_count") / F.lit(row_count) * 100, 2))
        .orderBy(F.desc("invalid_count"), F.asc("rule_name"))
    )

    display(invalid_long)


In [0]:
# Class imbalance checks
# The target column is is_fraud. This cell shows prevalence and imbalance ratio.

if row_count == 0:
    print("The table is empty, so class balance cannot be profiled.")
else:
    class_summary = (
        audit_df.groupBy("is_fraud")
        .count()
        .withColumn("class_pct", F.round(F.col("count") / F.lit(row_count) * 100, 4))
        .orderBy(F.col("is_fraud").asc())
    )

    class_counts = {row["is_fraud"]: row["count"] for row in class_summary.collect()}

    if len(class_counts) < 2:
        print("Only one class is present in is_fraud. This is a severe class imbalance issue for modeling.")
        imbalance_ratio = None
    else:
        minority_count = min(class_counts.values())
        majority_count = max(class_counts.values())
        imbalance_ratio = round(majority_count / minority_count, 2) if minority_count else None

    display(class_summary)
    print(f"Imbalance ratio (majority / minority): {imbalance_ratio}")


In [0]:
later_captured_features = ["transaction_status", "account_balance"]
doubted_features = ["transaction_type", "kyc_status"]

eligible_numeric_features = [c for c in numeric_features if c not in later_captured_features]

corr_df = spark.createDataFrame(
    [(c, audit_df.stat.corr(c, "is_fraud")) for c in eligible_numeric_features],
    ["feature", "correlation_with_is_fraud"],
).orderBy(F.desc(F.abs(F.col("correlation_with_is_fraud"))))

display(corr_df)

for c in doubted_features:
    display(audit_df.groupBy(c, "is_fraud").count().orderBy(F.desc("count")))



In [0]:
# Privacy risk checks
# This cell flags direct identifiers, high-cardinality quasi-identifiers, and sensitive attributes.

privacy_keywords = [
    "name",
    "email",
    "phone",
    "address",
    "ssn",
    "aadhaar",
    "pan",
    "customer",
    "account",
    "transaction",
    "loan",
    "credit",
    "kyc",
    "state",
    "date",
    "time",
]

privacy_named_columns = [
    column_name
    for column_name in audit_df.columns
    if any(keyword in column_name.lower() for keyword in privacy_keywords)
]

if row_count == 0:
    print("The table is empty, so privacy profiling cannot be computed.")
else:
    privacy_summary = audit_df.agg(
        *[F.countDistinct(F.col(column_name)).alias(column_name) for column_name in audit_df.columns]
    )

    stack_expr = "stack({0}, {1}) as (column_name, distinct_count)".format(
        len(audit_df.columns),
        ", ".join([f"'{column_name}', `{column_name}`" for column_name in audit_df.columns]),
    )

    privacy_long = (
        privacy_summary
        .selectExpr(stack_expr)
        .withColumn("distinct_ratio", F.round(F.col("distinct_count") / F.lit(row_count), 4))
        .withColumn(
            "risk_reason",
            F.when(F.col("column_name").rlike("_id$"), F.lit("Likely identifier"))
            .when(F.col("column_name").isin(privacy_named_columns), F.lit("Sensitive or quasi-identifier by name"))
            .when(F.col("distinct_count") == row_count, F.lit("Unique per row"))
            .when(F.col("distinct_ratio") >= 0.90, F.lit("Very high cardinality"))
            .otherwise(F.lit("Lower immediate risk")),
        )
        .orderBy(F.desc("distinct_ratio"), F.asc("column_name"))
    )

    display(privacy_long.filter(F.col("risk_reason") != "Lower immediate risk"))


In [0]:
# Numerical distribution analysis
# Uses the notebook's numeric feature list and keeps only columns present in audit_df.

available_numeric_features = [c for c in numeric_features if c in audit_df.columns]

if row_count == 0:
    print("The table is empty, so numerical distributions cannot be profiled.")
elif not available_numeric_features:
    print("No numeric features are available for distribution analysis.")
else:
    numeric_distribution = (
        audit_df.select(*available_numeric_features)
        .summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max")
    )

    numeric_missing = audit_df.agg(
        *[
            F.sum(F.when(F.col(column_name).isNull(), 1).otherwise(0)).alias(column_name)
            for column_name in available_numeric_features
        ]
    )

    missing_stack_expr = "stack({0}, {1}) as (feature, missing_count)".format(
        len(available_numeric_features),
        ", ".join([f"'{column_name}', `{column_name}`" for column_name in available_numeric_features]),
    )

    numeric_missing_long = (
        numeric_missing
        .selectExpr(missing_stack_expr)
        .withColumn("missing_pct", F.round(F.col("missing_count") / F.lit(row_count) * 100, 2))
        .orderBy(F.desc("missing_pct"), F.asc("feature"))
    )

    display(numeric_distribution)
    display(numeric_missing_long)


In [0]:
# Categorical-value analysis
# Derives categorical columns from the current DataFrame schema so the analysis stays aligned with audit_df.

available_categorical_features = [
    column_name
    for column_name, data_type in audit_df.dtypes
    if data_type == "string" and column_name not in identifiers
]

if row_count == 0:
    print("The table is empty, so categorical values cannot be profiled.")
elif not available_categorical_features:
    print("No categorical features are available for analysis.")
else:
    categorical_cardinality = spark.createDataFrame(
        [
            (
                column_name,
                audit_df.filter(F.col(column_name).isNull() | (F.trim(F.col(column_name)) == "")).count(),
                audit_df.filter(F.col(column_name).isNotNull() & (F.trim(F.col(column_name)) != "")).select(column_name).distinct().count(),
            )
            for column_name in available_categorical_features
        ],
        ["feature", "missing_or_blank_count", "distinct_non_blank_values"],
    ).withColumn(
        "missing_or_blank_pct",
        F.round(F.col("missing_or_blank_count") / F.lit(row_count) * 100, 2),
    ).orderBy(F.desc("distinct_non_blank_values"), F.asc("feature"))

    display(categorical_cardinality)

    for column_name in available_categorical_features:
        print(f"Top values for {column_name}")
        display(
            audit_df.groupBy(column_name)
            .count()
            .withColumn("pct", F.round(F.col("count") / F.lit(row_count) * 100, 2))
            .orderBy(F.desc("count"), F.asc(column_name))
            .limit(10)
        )


In [0]:
# Outlier inspection
# Uses an IQR rule to summarize numeric outliers and shows example records for flagged features.

available_numeric_features = [c for c in numeric_features if c in audit_df.columns]

if row_count == 0:
    print("The table is empty, so outliers cannot be inspected.")
elif not available_numeric_features:
    print("No numeric features are available for outlier inspection.")
else:
    quantiles = audit_df.approxQuantile(available_numeric_features, [0.25, 0.75], 0.01)
    outlier_rows = []

    for column_name, bounds in zip(available_numeric_features, quantiles):
        if len(bounds) < 2:
            outlier_rows.append((column_name, None, None, None, None, None, None))
            continue

        q1, q3 = bounds
        iqr = q3 - q1
        lower_bound = q1 - (1.5 * iqr)
        upper_bound = q3 + (1.5 * iqr)

        feature_outliers = audit_df.filter(
            F.col(column_name).isNotNull()
            & ((F.col(column_name) < F.lit(lower_bound)) | (F.col(column_name) > F.lit(upper_bound)))
        )

        outlier_count = feature_outliers.count()
        outlier_rows.append(
            (
                column_name,
                q1,
                q3,
                lower_bound,
                upper_bound,
                outlier_count,
                round((outlier_count / row_count) * 100, 2),
            )
        )

        if outlier_count > 0:
            print(f"Example outliers for {column_name}")
            display(
                feature_outliers
                .select("transaction_id", column_name, "is_fraud", "transaction_type", "account_type")
                .orderBy(F.desc(F.abs(F.col(column_name))))
                .limit(10)
            )

    outlier_summary = spark.createDataFrame(
        outlier_rows,
        [
            "feature",
            "q1",
            "q3",
            "lower_bound",
            "upper_bound",
            "outlier_count",
            "outlier_pct",
        ],
    ).orderBy(F.desc("outlier_count"), F.asc("feature"))

    display(outlier_summary)


### Data Audition Report

* **Schema checks**: The schema is perfect. 
* **Missing values**: No missing value found. 
* **Duplicates**: No Duplicate Row, No duplicated Identifiers
* **Invalid values**: Even if there is `has_loan` the `loan_type` shows "None" in some records
* **Class imbalance**: `Legits` are 99.1% and `Frauds` are 0.89%. Highly imbalanced
* **Target leakage**: `account_balance` and `transaction_status` are post transaction features. else there is no target lekage.
* **Privacy risks**: No alarming senesetive information.
* **For detail report go to the dashboards**

In [0]:
feature_audit = [
    ("transaction_id", "text", audit_df.select("transaction_id").distinct().count(), "High-cardinality identifier", "Likely ID column", "Drop from modeling"),
    ("customer_id", "text", audit_df.select("customer_id").distinct().count(), "High-cardinality identifier", "Likely customer identifier", "Drop from modeling"),
    ("transaction_date", "datetime", audit_df.select("transaction_date").distinct().count(), "Date-based patterns/seasonality may exist", "Potential temporal leakage if split incorrectly", "Keep; use time-based features/split"),
    ("transaction_time", "datetime", audit_df.select("transaction_time").distinct().count(), "Time-of-day behavior may be important", "Inconsistent granularity possible", "Extract useful time parts"),
    ("account_type", "categorical", audit_df.select("account_type").distinct().count(), "Categorical distribution should be checked for imbalance", "Possible rare categories / blanks", "Keep; encode and clean categories"),
    ("transaction_type", "categorical", audit_df.select("transaction_type").distinct().count(), "Categorical distribution may be imbalanced", "Potential target leakage concern", "Review before modeling"),
    ("transaction_amount", "numeric", audit_df.select("transaction_amount").distinct().count(), "May be right-skewed", "Possible outliers", "Keep; consider scaling/log transform"),
    ("transaction_direction", "categorical", audit_df.select("transaction_direction").distinct().count(), "Usually low-cardinality", "Unexpected labels possible", "Standardize categories"),
    ("account_balance", "numeric", audit_df.select("account_balance").distinct().count(), "May be skewed", "Potential leakage / negative values", "Review carefully; maybe drop"),
    ("merchant_category", "categorical", audit_df.select("merchant_category").distinct().count(), "Can have long-tail categories", "Rare labels / inconsistent naming", "Group rare categories"),
    ("state", "categorical", audit_df.select("state").distinct().count(), "Geographic imbalance may exist", "Inconsistent abbreviations/formats", "Standardize values"),
    ("credit_score", "numeric", audit_df.select("credit_score").distinct().count(), "Typically bounded numeric distribution", "Out-of-range values possible", "Keep; validate range"),
    ("has_loan", "categorical", audit_df.select("has_loan").distinct().count(), "Binary distribution may be imbalanced", "Non-binary values possible", "Keep; enforce binary format"),
    ("loan_type", "categorical", audit_df.select("loan_type").distinct().count(), "Sparse when no loan exists", "Inconsistent with has_loan", "Clean with business rules"),
    ("emi_amount", "numeric", audit_df.select("emi_amount").distinct().count(), "May be zero-inflated/skewed", "Inconsistent with has_loan", "Keep; validate conditional logic"),
    ("transaction_status", "categorical", audit_df.select("transaction_status").distinct().count(), "Status distribution may be imbalanced", "Potential post-event leakage", "Review; likely drop from modeling"),
    ("channel", "categorical", audit_df.select("channel").distinct().count(), "Channel usage may be imbalanced", "Inconsistent naming possible", "Standardize and encode"),
    ("kyc_status", "categorical", audit_df.select("kyc_status").distinct().count(), "May be imbalanced", "Potential policy/process leakage", "Review before modeling"),
    ("is_fraud", "categorical", audit_df.select("is_fraud").distinct().count(), "Highly imbalanced target", "Class imbalance", "Keep as target; handle imbalance"),
    ("transaction_hour", "numeric", audit_df.select("transaction_hour").distinct().count(), "Hourly pattern may show seasonality", "Invalid hour values possible", "Keep; validate 0-23 range"),
]

feature_audit_df = spark.createDataFrame(
    feature_audit,
    [
        "Feature Name",
        "Data Type",
        "Unique Values",
        "Distribution Notes",
        "Potential Issues",
        "Action",
    ],
)

display(feature_audit_df)
